In [1]:
from random import choice

import duckdb
import mlflow
import polars as pl
from sklearn.metrics import classification_report

from fantasy_football.constants import MLFLOW_TRACKING_URI
from fantasy_football.extraction.availability import (
    load_player_availability_data,
)
from fantasy_football.features.availability import (
    add_chance_of_playing,
    add_positional_availability,
)
from fantasy_football.storage.database import (
    get_connection,
    load_player_availability,
    load_player_match,
    load_player_week,
)

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment("minutes_played_classification")

/Users/jamie/personal/fantasy_football/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<Experiment: artifact_location='/Users/jamie/personal/fantasy_football/notebooks/mlruns/5', creation_time=1782214959387, effective_trace_archival_retention=None, experiment_id='5', last_update_time=1782214959387, lifecycle_stage='active', name='minutes_played_classification', tags={}, trace_location=None, workspace='default'>

In [2]:
import numpy as np
from sklearn.metrics import (
    brier_score_loss,
    log_loss,
    roc_auc_score,
)

BUCKETS = ["0_minutes", "1_to_59_minutes", "60_minutes_plus"]


def buckets_to_proba(bucket_preds, classes=BUCKETS) -> np.ndarray:
    """Turn hard bucket predictions into degenerate one-hot probabilities.

    Lets a baseline that only emits a class be scored on the same probabilistic
    axis as a calibrated model. It's honest: a confidently-wrong baseline earns a
    brutal log loss, which is exactly the point of putting it on this axis.
    """
    ci = {c: i for i, c in enumerate(classes)}
    proba = np.zeros((len(bucket_preds), len(classes)))
    for row, bucket in enumerate(bucket_preds):
        proba[row, ci[bucket]] = 1.0
    return proba


def boundary_metrics(y_true_bucket, proba, classes, true_minutes) -> dict:
    """Two-boundary scores for a minutes model, returned as a plain dict (no I/O).

    Points modelling consumes P(min>0) and P(min>=60) -- the latter carries the
    2nd appearance point AND clean-sheet eligibility -- and then *multiplies* them,
    so we score the probabilities themselves (log loss, Brier, AUC) rather than the
    argmax, and add expected-minutes / expected-appearance-point error as the
    leverage metric. The 1-59 bucket is the residual P(>0) - P(>=60), never scored.

    Boundaries are derived from the 3-class `proba`, indexed by `classes` (e.g.
    model.classes_) not column position, so label ordering can't bite. Returns a
    dict so callers can aggregate across CV folds before logging anything.
    """
    classes = list(classes)
    y_true_bucket = np.asarray(y_true_bucket)
    true_minutes = np.asarray(true_minutes, dtype=float)
    ci = {c: i for i, c in enumerate(classes)}

    # Clip to [0, 1]: p_appear sums two proba columns, so floating-point rounding
    # can nudge it a hair past 1.0, which brier_score_loss rejects outright.
    p_appear = np.clip(
        proba[:, ci["1_to_59_minutes"]] + proba[:, ci["60_minutes_plus"]],
        0.0,
        1.0,
    )
    p_60 = np.clip(proba[:, ci["60_minutes_plus"]], 0.0, 1.0)

    y_appear = (y_true_bucket != "0_minutes").astype(int)
    y_60 = (y_true_bucket == "60_minutes_plus").astype(int)

    metrics = {
        "logloss_appear": log_loss(y_appear, p_appear, labels=[0, 1]),
        "brier_appear": brier_score_loss(y_appear, p_appear),
        "logloss_60": log_loss(y_60, p_60, labels=[0, 1]),
        "brier_60": brier_score_loss(y_60, p_60),
    }
    # AUC needs both classes present in the truth vector.
    if y_appear.min() != y_appear.max():
        metrics["auc_appear"] = roc_auc_score(y_appear, p_appear)
    if y_60.min() != y_60.max():
        metrics["auc_60"] = roc_auc_score(y_60, p_60)

    # Points-leverage: representative minutes per bucket mass, plus expected
    # appearance points (1 for >0, +1 for >=60), both MAE vs realised.
    e_min = p_60 * 75 + (p_appear - p_60) * 30
    e_app = p_appear + p_60
    true_app = y_appear + y_60
    metrics["e_min_mae"] = float(np.mean(np.abs(e_min - true_minutes)))
    metrics["e_app_mae"] = float(np.mean(np.abs(e_app - true_app)))
    return metrics


def log_boundary_report(y_true_bucket, proba, classes, true_minutes) -> dict:
    """Score on the two boundaries and log to the active MLflow run.

    Thin wrapper over boundary_metrics: logs each metric, plus the 3-class report
    as a diagnostic artifact (the 1-59 class is informational, never optimised).
    Call inside an active `mlflow.start_run()`. Returns the metric dict.
    """
    metrics = boundary_metrics(y_true_bucket, proba, classes, true_minutes)
    for name, value in metrics.items():
        mlflow.log_metric(name, value)

    classes = list(classes)
    hard_pred = np.asarray(classes)[proba.argmax(axis=1)]
    report = classification_report(
        np.asarray(y_true_bucket), hard_pred, output_dict=True, zero_division=0
    )
    mlflow.log_dict(report, "class_report.json")
    return metrics

In [3]:
conn = get_connection()

In [4]:
player_match = load_player_match()
player_match.head()

season,gw,element,opponent,is_home,minutes,total_points
str,i64,i64,i64,bool,i64,i64
"""2016-17""",1,6,9,true,90,0
"""2016-17""",1,7,9,true,0,0
"""2016-17""",1,11,9,true,90,6
"""2016-17""",1,13,9,true,90,5
"""2016-17""",1,14,9,true,0,0


In [6]:
def create_buckets(
    player_data: pl.DataFrame, column_to_bucket: str = "minutes"
) -> pl.DataFrame:
    player_data = player_data.with_columns(
        pl.when(pl.col(column_to_bucket) == 0)
        .then(pl.lit("0_minutes"))
        .when(pl.col(column_to_bucket) < 60)
        .then(pl.lit("1_to_59_minutes"))
        .otherwise(pl.lit("60_minutes_plus"))
        .alias("minutes_bucket")
    )
    return player_data

In [7]:
player_match = create_buckets(player_match)
player_match.head()

season,gw,element,opponent,is_home,minutes,total_points,minutes_bucket
str,i64,i64,i64,bool,i64,i64,str
"""2016-17""",1,6,9,true,90,0,"""60_minutes_plus"""
"""2016-17""",1,7,9,true,0,0,"""0_minutes"""
"""2016-17""",1,11,9,true,90,6,"""60_minutes_plus"""
"""2016-17""",1,13,9,true,90,5,"""60_minutes_plus"""
"""2016-17""",1,14,9,true,0,0,"""0_minutes"""


In [8]:
train_seasons = ["2022-23", "2023-24", "2024-25"]
test_season = "2025-26"

train = player_match.filter(pl.col("season").is_in(train_seasons))
test = player_match.filter(pl.col("season") == test_season)
test_y = create_buckets(test)
test_y.head()

season,gw,element,opponent,is_home,minutes,total_points,minutes_bucket
str,i64,i64,i64,bool,i64,i64,str
"""2025-26""",1,1,14,false,90,10,"""60_minutes_plus"""
"""2025-26""",1,2,14,false,0,0,"""0_minutes"""
"""2025-26""",1,3,14,false,0,0,"""0_minutes"""
"""2025-26""",1,4,14,false,0,0,"""0_minutes"""
"""2025-26""",1,5,14,false,90,6,"""60_minutes_plus"""


In [9]:
# Score every model on the SAME test season so the boundary metrics line up.
# Sits after the split above so test_season is defined; the random baseline only
# needs minutes_bucket/minutes, both present this early.
test_rows = player_match.filter(pl.col("season") == test_season)
y_true = test_rows["minutes_bucket"].to_numpy()
true_minutes = test_rows["minutes"].to_numpy()

random_pred = [choice(BUCKETS) for _ in range(len(test_rows))]
random_proba = buckets_to_proba(random_pred)

with mlflow.start_run(
    run_name="random_baseline",
    description="Random bucket guess, scored on the two point boundaries",
):
    metrics = log_boundary_report(y_true, random_proba, BUCKETS, true_minutes)
print(metrics)

{'logloss_appear': 19.40495206329079, 'brier_appear': np.float64(0.5383736175076478), 'logloss_60': 15.231948322674276, 'brier_60': np.float64(0.4225972366961374), 'auc_appear': np.float64(0.49910235865555863), 'auc_60': np.float64(0.49894728867378935), 'e_min_mae': 38.69297744310351, 'e_app_mae': 0.9609708542037853}


In [10]:
def create_rolling_minutes(
    player_data: pl.DataFrame, window: int = 3
) -> pl.DataFrame:
    # Rolling mean of each player's *prior* gameweeks.
    #   - sort by element, season, gw then .over("element") so every window stays
    #     inside one player in chronological order. (The raw frame is ordered
    #     season, gw, element, so a plain rolling_mean blends three players together.)
    #     polars 0.20.x has no order_by= on .over(), hence the explicit sort.
    #   - .shift(1) drops the current match, so we never leak this week's minutes
    #     into the feature predicting this week's bucket.
    #   - _orig restores the original row order so downstream frames stay aligned.
    return (
        player_data.with_row_index("_orig")
        .sort(["element", "season", "gw"])
        .with_columns(
            pl.col("minutes")
            .shift(1)
            .rolling_mean(window_size=window, min_periods=1)
            .over("element")
            .alias("rolling_minutes")
        )
        .sort("_orig")
        .drop("_orig")
    )


train = create_rolling_minutes(train)
train.head()

season,gw,element,opponent,is_home,minutes,total_points,minutes_bucket,rolling_minutes
str,i64,i64,i64,bool,i64,i64,str,f64
"""2022-23""",1,1,7,false,0,0,"""0_minutes""",null
"""2022-23""",1,2,12,true,0,0,"""0_minutes""",null
"""2022-23""",1,3,7,false,90,2,"""60_minutes_plus""",null
"""2022-23""",1,4,7,false,0,0,"""0_minutes""",null
"""2022-23""",1,5,7,false,0,0,"""0_minutes""",null


In [11]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median").set_output(transform="polars")
train = train.with_columns(
    imputer.fit_transform(train.select("rolling_minutes"))
)
train.head()

season,gw,element,opponent,is_home,minutes,total_points,minutes_bucket,rolling_minutes
str,i64,i64,i64,bool,i64,i64,str,f64
"""2022-23""",1,1,7,false,0,0,"""0_minutes""",1.0
"""2022-23""",1,2,12,true,0,0,"""0_minutes""",1.0
"""2022-23""",1,3,7,false,90,2,"""60_minutes_plus""",1.0
"""2022-23""",1,4,7,false,0,0,"""0_minutes""",1.0
"""2022-23""",1,5,7,false,0,0,"""0_minutes""",1.0


In [12]:
test = create_rolling_minutes(test)
# Keep rolling_minutes as a column (don't replace the whole frame) so the bucket
# step below still has season/gw/element/minutes alongside it. Imputer was fit on
# train only — correct, no test leakage.
test = test.with_columns(imputer.transform(test.select("rolling_minutes")))
test.head()

season,gw,element,opponent,is_home,minutes,total_points,minutes_bucket,rolling_minutes
str,i64,i64,i64,bool,i64,i64,str,f64
"""2025-26""",1,1,14,false,90,10,"""60_minutes_plus""",1.0
"""2025-26""",1,2,14,false,0,0,"""0_minutes""",1.0
"""2025-26""",1,3,14,false,0,0,"""0_minutes""",1.0
"""2025-26""",1,4,14,false,0,0,"""0_minutes""",1.0
"""2025-26""",1,5,14,false,90,6,"""60_minutes_plus""",1.0


In [13]:
train_predictions = create_buckets(train, "rolling_minutes")
train_predictions.head()

season,gw,element,opponent,is_home,minutes,total_points,minutes_bucket,rolling_minutes
str,i64,i64,i64,bool,i64,i64,str,f64
"""2022-23""",1,1,7,false,0,0,"""1_to_59_minutes""",1.0
"""2022-23""",1,2,12,true,0,0,"""1_to_59_minutes""",1.0
"""2022-23""",1,3,7,false,90,2,"""1_to_59_minutes""",1.0
"""2022-23""",1,4,7,false,0,0,"""1_to_59_minutes""",1.0
"""2022-23""",1,5,7,false,0,0,"""1_to_59_minutes""",1.0


In [14]:
test_predictions = create_buckets(test, "rolling_minutes")
test_predictions.head()

season,gw,element,opponent,is_home,minutes,total_points,minutes_bucket,rolling_minutes
str,i64,i64,i64,bool,i64,i64,str,f64
"""2025-26""",1,1,14,false,90,10,"""1_to_59_minutes""",1.0
"""2025-26""",1,2,14,false,0,0,"""1_to_59_minutes""",1.0
"""2025-26""",1,3,14,false,0,0,"""1_to_59_minutes""",1.0
"""2025-26""",1,4,14,false,0,0,"""1_to_59_minutes""",1.0
"""2025-26""",1,5,14,false,90,6,"""1_to_59_minutes""",1.0


In [15]:
# Trailing-mean minutes, bucketed. test_predictions aligns row-for-row with
# player_match.filter(test_season) — create_rolling_minutes restores original order.
test_rows = player_match.filter(pl.col("season") == test_season)
y_true = test_rows["minutes_bucket"].to_numpy()
true_minutes = test_rows["minutes"].to_numpy()

rolling_proba = buckets_to_proba(test_predictions["minutes_bucket"].to_list())

with mlflow.start_run(
    run_name="rolling_minutes_baseline",
    description="Trailing 3-gw mean minutes bucketed, scored on the two point boundaries",
):
    metrics = log_boundary_report(y_true, rolling_proba, BUCKETS, true_minutes)
print(metrics)

{'logloss_appear': 5.30228349853016, 'brier_appear': np.float64(0.1471072713214778), 'logloss_60': 4.617687937134012, 'brier_60': np.float64(0.12811375937069283), 'auc_appear': np.float64(0.8694395624960732), 'auc_60': np.float64(0.814039152702397), 'e_min_mae': 14.244528860053114, 'e_app_mae': 0.27522103069217063}


In [16]:
def load_value_features(conn: duckdb.DuckDBPyConnection) -> pl.DataFrame:
    query = """
    SELECT
        season,
        gw,
        element,
        team,
        position,
        value,
        value::DOUBLE / SUM(value) OVER (PARTITION BY season, gw, team)          AS value_vs_team,
        RANK()  OVER (PARTITION BY season, gw, team, position ORDER BY value DESC) AS pos_value_rank,
        COUNT(*) OVER (PARTITION BY season, gw, team, position)                   AS players_same_pos
    FROM player_week
    """
    return conn.execute(query).pl()


value_features = load_value_features(conn)
value_features.head()

season,gw,element,team,position,value,value_vs_team,pos_value_rank,players_same_pos
str,i64,i64,str,str,i64,f64,i64,i64
"""2025-26""",37,756,"""Bournemouth""","""MID""",45,0.021541,14,21
"""2025-26""",37,95,"""Bournemouth""","""MID""",44,0.021063,20,21
"""2025-26""",37,687,"""Bournemouth""","""MID""",43,0.020584,21,21
"""2025-26""",37,157,"""Brighton""","""MID""",61,0.027318,1,23
"""2025-26""",37,159,"""Brighton""","""MID""",58,0.025974,2,23


In [17]:
player_match = player_match.join(
    value_features,
    on=["season", "gw", "element"],
    how="left",
)
player_match.head()

/var/folders/ws/c0kbfc596sgcz0f3y4dzqtyc0000gn/T/ipykernel_35442/3497995312.py:1: DeprecationWarning: The default coalesce behavior of left join will change to `False` in the next breaking release. Pass `coalesce=True` to keep the current behavior and silence this warning.
  player_match = player_match.join(


season,gw,element,opponent,is_home,minutes,total_points,minutes_bucket,team,position,value,value_vs_team,pos_value_rank,players_same_pos
str,i64,i64,i64,bool,i64,i64,str,str,str,i64,f64,i64,i64
"""2016-17""",1,6,9,true,90,0,"""60_minutes_plus""",null,"""DEF""",65,0.005745,1,76
"""2016-17""",1,7,9,true,0,0,"""0_minutes""",null,"""DEF""",50,0.004419,28,76
"""2016-17""",1,11,9,true,90,6,"""60_minutes_plus""",null,"""DEF""",45,0.003977,49,76
"""2016-17""",1,13,9,true,90,5,"""60_minutes_plus""",null,"""MID""",75,0.006628,11,71
"""2016-17""",1,14,9,true,0,0,"""0_minutes""",null,"""MID""",95,0.008396,2,71


In [18]:
# player_availability is sourced from the Randdalf/fplcache snapshots (2022-23 on).
# Populate it once if empty; the first run is slow (~a bootstrap snapshot per
# gameweek per season), reruns are cheap (immutable seasons skip, current upserts).
if load_player_availability(conn).is_empty():
    load_player_availability_data(conn)

availability = load_player_availability(conn)
player_match = add_chance_of_playing(player_match, availability)

# Count fit same-position rivals. Runs after add_chance_of_playing so null
# chances are already filled to 100 (= fit). Needs team/position/value, all
# present from the value-features join above.
player_match = add_positional_availability(player_match)
player_match.head()

season,gw,element,opponent,is_home,minutes,total_points,minutes_bucket,team,position,value,value_vs_team,pos_value_rank,players_same_pos,chance_of_playing_this_round,fit_rivals_same_pos,fit_rivals_ahead
str,i64,i64,i64,bool,i64,i64,str,str,str,i64,f64,i64,i64,i64,i32,i32
"""2016-17""",1,6,9,true,90,0,"""60_minutes_plus""",null,"""DEF""",65,0.005745,1,76,100,75,null
"""2016-17""",1,7,9,true,0,0,"""0_minutes""",null,"""DEF""",50,0.004419,28,76,100,75,null
"""2016-17""",1,11,9,true,90,6,"""60_minutes_plus""",null,"""DEF""",45,0.003977,49,76,100,75,null
"""2016-17""",1,13,9,true,90,5,"""60_minutes_plus""",null,"""MID""",75,0.006628,11,71,100,70,null
"""2016-17""",1,14,9,true,0,0,"""0_minutes""",null,"""MID""",95,0.008396,2,71,100,70,null


In [19]:
features = [
    "position",
    "value",
    "value_vs_team",
    "pos_value_rank",
    "players_same_pos",
    "chance_of_playing_this_round",
    "fit_rivals_same_pos",
    "fit_rivals_ahead",
]
train_X = player_match.filter(pl.col("season").is_in(train_seasons)).select(
    features
)
train_y = player_match.filter(pl.col("season").is_in(train_seasons)).select(
    "minutes_bucket"
)

test_X = player_match.filter(pl.col("season") == test_season).select(features)
test_y = player_match.filter(pl.col("season") == test_season).select(
    "minutes_bucket"
)

In [20]:
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(sparse_output=False).set_output(transform="polars")
train_pos = ohe.fit_transform(train_X.select("position"))
test_pos = ohe.transform(test_X.select("position"))

train_X = train_X.drop("position")
train_X = pl.concat([train_X, train_pos], how="horizontal")

test_X = test_X.drop("position")
test_X = pl.concat([test_X, test_pos], how="horizontal")

train_X.head()

value,value_vs_team,pos_value_rank,players_same_pos,chance_of_playing_this_round,fit_rivals_same_pos,fit_rivals_ahead,position_AM,position_DEF,position_FWD,position_GK,position_MID
i64,f64,i64,i64,i64,i32,i32,f64,f64,f64,f64,f64
45,0.030612,4,10,100,8,3,0.0,1.0,0.0,0.0,0.0
45,0.033582,1,3,100,2,0,0.0,0.0,0.0,1.0,0.0
50,0.034014,7,14,100,13,6,0.0,0.0,0.0,0.0,1.0
45,0.030612,12,14,100,13,11,0.0,0.0,0.0,0.0,1.0
45,0.030612,4,10,100,8,3,0.0,1.0,0.0,0.0,0.0


In [21]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Scale first: value/rank features are on wildly different magnitudes, which made
# the unscaled solver overflow and mis-converge — and since we now score on
# calibration (log loss / Brier) the probabilities, not just the argmax, must be
# sane. Scaling the one-hot position columns too is harmless.
lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
lr.fit(train_X, train_y.to_numpy().ravel())

pred = lr.predict(test_X)  # kept for the error-analysis cells below
proba = lr.predict_proba(test_X)

test_rows = player_match.filter(pl.col("season") == test_season)
y_true = test_y["minutes_bucket"].to_numpy()
true_minutes = test_rows["minutes"].to_numpy()

with mlflow.start_run(
    run_name="logistic_regression",
    description="Scaled LR on value/availability features, scored on the two point boundaries",
):
    metrics = log_boundary_report(y_true, proba, lr.classes_, true_minutes)
print(metrics)

/Users/jamie/personal/fantasy_football/.venv/lib/python3.12/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/Users/jamie/personal/fantasy_football/.venv/lib/python3.12/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/Users/jamie/personal/fantasy_football/.venv/lib/python3.12/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/Users/jamie/personal/fantasy_football/.venv/lib/python3.12/site-packages/sklearn/linear_model/_linear_loss.py:336: RuntimeWarning: divide by zero encountered in matmul
  grad[:, :n_features] = grad_pointwise.T @ X + l2_reg_strength * weights
/Users/jamie/personal/fant

{'logloss_appear': 0.3479797145923608, 'brier_appear': np.float64(0.10977904077813065), 'logloss_60': 0.3668745610502938, 'brier_60': np.float64(0.12249889421775902), 'auc_appear': np.float64(0.9244720274130179), 'auc_60': np.float64(0.8833466294885494), 'e_min_mae': 20.414023919269727, 'e_app_mae': 0.4398389386209456}


In [22]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier()
rf.fit(train_X, train_y.to_numpy().ravel())

pred = rf.predict(test_X)  # kept for the error-analysis cells below
proba = rf.predict_proba(test_X)

test_rows = player_match.filter(pl.col("season") == test_season)
y_true = test_y["minutes_bucket"].to_numpy()
true_minutes = test_rows["minutes"].to_numpy()

with mlflow.start_run(
    run_name="random_forest",
    description="RF on value/availability features, scored on the two point boundaries",
):
    metrics = log_boundary_report(y_true, proba, rf.classes_, true_minutes)
print(metrics)

{'logloss_appear': 0.43575046119784777, 'brier_appear': np.float64(0.11385697116003721), 'logloss_60': 0.4423825758129776, 'brier_60': np.float64(0.12777288860312977), 'auc_appear': np.float64(0.9125479094313333), 'auc_60': np.float64(0.8716183632288019), 'e_min_mae': 19.485110033425187, 'e_app_mae': 0.4136312122760768}


## Error analysis — confidently wrong, far-off predictions

`pred` aligns row-for-row with `test_X`, which is `player_match` filtered to the
test season in original order. So we can glue predictions back onto the test rows
by position, attach `predict_proba` confidence, score each row by how many buckets
it missed by, then join `player_week` for names to see *who* the model got wrong.

In [23]:
# Ordinal rank of each bucket so we can measure "how far off" a miss was.
# Built with when/then (not replace_strict) to stay safe on polars 0.20.x.
def bucket_ord(col: str) -> pl.Expr:
    return (
        pl.when(pl.col(col) == "0_minutes")
        .then(0)
        .when(pl.col(col) == "1_to_59_minutes")
        .then(1)
        .otherwise(2)
    )


# Pick the model to analyse: rf is the most recent fit; swap to lr if you prefer.
model = rf
test_rows = player_match.filter(pl.col("season") == test_season)

# Confidence the model placed on the class it actually predicted.
proba = model.predict_proba(test_X)
pred_confidence = proba.max(axis=1)

errors = (
    test_rows
    # keys + the exact features the model trained on (`features`) + the target,
    # so every row shows the inputs that drove its (mis)prediction.
    .select(
        "season",
        "gw",
        "element",
        *features,
        "minutes",
        "minutes_bucket",
    )
    .with_columns(
        pl.Series("predicted_bucket", pred),
        pl.Series("predicted_confidence", pred_confidence),
    )
    .with_columns(
        (bucket_ord("predicted_bucket") - bucket_ord("minutes_bucket"))
        .abs()
        .alias("bucket_distance")
    )
)
errors.head()

season,gw,element,position,value,value_vs_team,pos_value_rank,players_same_pos,chance_of_playing_this_round,fit_rivals_same_pos,fit_rivals_ahead,minutes,minutes_bucket,predicted_bucket,predicted_confidence,bucket_distance
str,i64,i64,str,i64,f64,i64,i64,i64,i32,i32,i64,str,str,f64,i32
"""2025-26""",1,1,"""GK""",60,0.02893,1,4,100,3,0,90,"""60_minutes_plus""","""60_minutes_plus""",0.88,0
"""2025-26""",1,2,"""GK""",41,0.019769,2,4,100,3,1,0,"""0_minutes""","""0_minutes""",0.985,0
"""2025-26""",1,3,"""GK""",40,0.019286,3,4,100,3,2,0,"""0_minutes""","""0_minutes""",1.0,0
"""2025-26""",1,4,"""GK""",39,0.018804,4,4,100,3,3,0,"""0_minutes""","""0_minutes""",0.978333,0
"""2025-26""",1,5,"""DEF""",62,0.029894,2,14,100,11,1,90,"""60_minutes_plus""","""60_minutes_plus""",0.847762,0


In [24]:
def bucket_ord(col):
    return (
        pl.when(pl.col(col) == "0_minutes")
        .then(0)
        .when(pl.col(col) == "1_to_59_minutes")
        .then(1)
        .otherwise(2)
    )


model = rf
test_rows = player_match.filter(pl.col("season") == test_season)
proba = model.predict_proba(test_X)
pred_confidence = proba.max(axis=1)

errors = (
    test_rows.select(
        "season", "gw", "element", "minutes", "minutes_bucket", *features
    )
    .with_columns(
        pl.Series("predicted_bucket", pred),
        pl.Series("predicted_confidence", pred_confidence),
    )
    .with_columns(
        (bucket_ord("predicted_bucket") - bucket_ord("minutes_bucket"))
        .abs()
        .alias("bucket_distance")
    )
)

names = load_player_week().select("season", "gw", "element", "name", "team")
errors = errors.join(names, on=["season", "gw", "element"], how="left")

worst = (
    errors.filter(pl.col("predicted_bucket") != pl.col("minutes_bucket"))
    .sort(["bucket_distance", "predicted_confidence"], descending=True)
    .select(
        "name",
        "team",
        "position",
        "gw",
        "value",
        "minutes",
        "minutes_bucket",
        "predicted_bucket",
        "predicted_confidence",
        "bucket_distance",
    )
)
print(errors.shape, "errors rows; worst preview:")
print(worst.head(10))

(29747, 18) errors rows; worst preview:
shape: (10, 10)
┌────────────┬────────────┬──────────┬─────┬───┬────────────┬────────────┬────────────┬────────────┐
│ name       ┆ team       ┆ position ┆ gw  ┆ … ┆ minutes_bu ┆ predicted_ ┆ predicted_ ┆ bucket_dis │
│ ---        ┆ ---        ┆ ---      ┆ --- ┆   ┆ cket       ┆ bucket     ┆ confidence ┆ tance      │
│ str        ┆ str        ┆ str      ┆ i64 ┆   ┆ ---        ┆ ---        ┆ ---        ┆ ---        │
│            ┆            ┆          ┆     ┆   ┆ str        ┆ str        ┆ f64        ┆ i32        │
╞════════════╪════════════╪══════════╪═════╪═══╪════════════╪════════════╪════════════╪════════════╡
│ Martin     ┆ Burnley    ┆ GK       ┆ 6   ┆ … ┆ 60_minutes ┆ 0_minutes  ┆ 1.0        ┆ 2          │
│ Dúbravka   ┆            ┆          ┆     ┆   ┆ _plus      ┆            ┆            ┆            │
│ Aaron      ┆ Newcastle  ┆ GK       ┆ 32  ┆ … ┆ 60_minutes ┆ 0_minutes  ┆ 1.0        ┆ 2          │
│ Ramsdale   ┆            ┆        

/var/folders/ws/c0kbfc596sgcz0f3y4dzqtyc0000gn/T/ipykernel_35442/2055862598.py:26: DeprecationWarning: The default coalesce behavior of left join will change to `False` in the next breaking release. Pass `coalesce=True` to keep the current behavior and silence this warning.
  errors = errors.join(names, on=["season", "gw", "element"], how="left")


In [25]:
errors.filter(pl.col("bucket_distance") == 2)["position"].value_counts()

position,count
str,u32
"""MID""",1161
"""DEF""",1430
"""GK""",395
"""FWD""",134


In [26]:
errors.filter(
    (pl.col("bucket_distance") == 2) & (pl.col("position") == "GK")
).tail()

season,gw,element,minutes,minutes_bucket,position,value,value_vs_team,pos_value_rank,players_same_pos,chance_of_playing_this_round,fit_rivals_same_pos,fit_rivals_ahead,predicted_bucket,predicted_confidence,bucket_distance,name,team
str,i64,i64,i64,str,str,i64,f64,i64,i64,i64,i32,i32,str,f64,i32,str,str
"""2025-26""",38,567,90,"""60_minutes_plus""","""GK""",39,0.017326,2,3,100,1,0,"""0_minutes""",0.570667,2,"""Antonín Kinský""","""Spurs"""
"""2025-26""",38,665,0,"""0_minutes""","""GK""",44,0.024747,1,4,100,3,0,"""60_minutes_plus""",0.6335,2,"""Lucas Estella Perri""","""Leeds"""
"""2025-26""",38,679,90,"""60_minutes_plus""","""GK""",42,0.02175,2,6,100,3,1,"""0_minutes""",0.889167,2,"""Mads Hermansen""","""West Ham"""
"""2025-26""",38,736,0,"""0_minutes""","""GK""",56,0.025676,1,4,100,2,0,"""60_minutes_plus""",0.89,2,"""Gianluigi Donnarumma""","""Man City"""
"""2025-26""",38,738,0,"""0_minutes""","""GK""",45,0.023304,1,6,100,3,0,"""60_minutes_plus""",0.924333,2,"""Lukasz Fabianski""","""West Ham"""


## Season-based cross-validation

A single test season is a noisy way to pick a model. Random k-fold would be worse
here: it leaks the future into the past and splits the same player across train and
test. So we use **expanding-window CV by season** -- train on all prior seasons,
test on the next -- which mirrors how the model is actually used.

All preprocessing (impute / scale / one-hot) lives inside the pipeline, so it is
refit on each fold's train split only and never sees the test season. We rank
models on **mean `logloss_60`** across folds (the high-leverage boundary), and read
the std as the error bar. One MLflow run per model; no throwaway fold models logged.


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Expanding-window folds: train on all prior seasons, test on the next.
SEASONS_ORDERED = ["2022-23", "2023-24", "2024-25", "2025-26"]


def season_folds(seasons=SEASONS_ORDERED, min_train=1):
    return [(seasons[:i], seasons[i]) for i in range(min_train, len(seasons))]


# One frame with every feature + target + trailing minutes (for the rolling
# baseline). rolling_minutes is NOT a model feature, so its first-gw nulls are
# harmless for the sklearn models; the rolling baseline imputes them per fold.
model_df = create_rolling_minutes(
    player_match.filter(pl.col("season").is_in(SEASONS_ORDERED))
)

CAT_FEATURES = ["position"]
NUM_FEATURES = [
    "value",
    "value_vs_team",
    "pos_value_rank",
    "players_same_pos",
    "chance_of_playing_this_round",
    "fit_rivals_same_pos",
    "fit_rivals_ahead",
]
FEATURES = NUM_FEATURES + CAT_FEATURES


def make_preprocessor(scale: bool) -> ColumnTransformer:
    # Imputer (and scaler) sit INSIDE the pipeline so they refit on each fold's
    # train split only -- never the test season. This is what keeps CV honest.
    num_steps = [("impute", SimpleImputer(strategy="median"))]
    if scale:
        num_steps.append(("scale", StandardScaler()))
    return ColumnTransformer(
        [
            ("num", Pipeline(num_steps), NUM_FEATURES),
            ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_FEATURES),
        ]
    )


def make_lr() -> Pipeline:
    return Pipeline(
        [
            ("prep", make_preprocessor(scale=True)),
            ("clf", LogisticRegression(max_iter=1000)),
        ]
    )


def make_rf() -> Pipeline:
    return Pipeline(
        [
            ("prep", make_preprocessor(scale=False)),
            ("clf", RandomForestClassifier(random_state=0)),
        ]
    )


def sklearn_predict_fn(make_estimator):
    """Fit on the train seasons, predict_proba on the held-out season."""

    def _fn(train_df, test_df):
        est = make_estimator()
        est.fit(
            train_df.select(FEATURES), train_df["minutes_bucket"].to_numpy()
        )
        return est.predict_proba(test_df.select(FEATURES)), list(est.classes_)

    return _fn


def rolling_predict_fn(train_df, test_df):
    """Bar to beat: bucket each player's trailing-mean minutes. Median impute is
    fit on the train split only, then applied to the held-out season.
    """
    median = train_df["rolling_minutes"].median()
    rm = test_df["rolling_minutes"].fill_null(median)
    labels = (
        test_df.with_columns(rm.alias("_rm"))
        .select(
            pl.when(pl.col("_rm") == 0)
            .then(pl.lit("0_minutes"))
            .when(pl.col("_rm") < 60)
            .then(pl.lit("1_to_59_minutes"))
            .otherwise(pl.lit("60_minutes_plus"))
            .alias("b")
        )["b"]
        .to_list()
    )
    return buckets_to_proba(labels), BUCKETS


def random_predict_fn(train_df, test_df):
    """Floor: random bucket per row, ignores the features entirely."""
    return buckets_to_proba(
        [choice(BUCKETS) for _ in range(len(test_df))]
    ), BUCKETS


def cross_val_boundary(run_name, predict_fn, folds, description=""):
    """One MLflow run per model. Logs every boundary metric per fold (as a
    step-indexed trace) plus its mean/std across folds. No per-fold model
    artifacts -- fold models are throwaway generalisation estimates.
    """
    per_fold = []
    with mlflow.start_run(run_name=run_name, description=description):
        mlflow.log_param("cv", "expanding_window_by_season")
        mlflow.log_param("n_folds", len(folds))
        for i, (train_seasons_i, test_season_i) in enumerate(folds):
            train_df = model_df.filter(pl.col("season").is_in(train_seasons_i))
            test_df = model_df.filter(pl.col("season") == test_season_i)
            proba, classes = predict_fn(train_df, test_df)
            m = boundary_metrics(
                test_df["minutes_bucket"].to_numpy(),
                proba,
                classes,
                test_df["minutes"].to_numpy(),
            )
            for k, v in m.items():
                mlflow.log_metric(k, v, step=i)
            per_fold.append(m)

        # Aggregate only metrics present in EVERY fold (AUC can drop out if a
        # fold's truth vector is single-class -- won't happen here, but be safe).
        shared = set(per_fold[0])
        for m in per_fold:
            shared &= set(m)
        agg = {}
        for k in sorted(shared):
            agg[f"{k}_mean"] = float(np.mean([m[k] for m in per_fold]))
            agg[f"{k}_std"] = float(np.std([m[k] for m in per_fold]))
        for k, v in agg.items():
            mlflow.log_metric(k, v)
    return per_fold, agg

In [ ]:
folds = season_folds()
print("folds:", folds)

cv_models = {
    "random_baseline": (random_predict_fn, "Random floor, season CV"),
    "rolling_minutes_baseline": (
        rolling_predict_fn,
        "Rolling-mean bar, season CV",
    ),
    "logistic_regression": (
        sklearn_predict_fn(make_lr),
        "Scaled LR, season CV",
    ),
    "random_forest": (sklearn_predict_fn(make_rf), "RF, season CV"),
}

cv_results = {}
for name, (fn, desc) in cv_models.items():
    _, agg = cross_val_boundary(f"cv_{name}", fn, folds, desc)
    cv_results[name] = agg

# Rank on mean logloss_60 (the high-leverage boundary); lower is better. std is
# the error bar across 3 folds -- treat small gaps as noise.
summary = pl.DataFrame(
    [
        {
            "model": name,
            "logloss_60_mean": agg["logloss_60_mean"],
            "logloss_60_std": agg["logloss_60_std"],
            "logloss_appear_mean": agg["logloss_appear_mean"],
            "e_min_mae_mean": agg["e_min_mae_mean"],
            "e_app_mae_mean": agg["e_app_mae_mean"],
        }
        for name, agg in cv_results.items()
    ]
).sort("logloss_60_mean")
summary